In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/tastelet/data")
RAW_DIR = DATA_DIR / "raw"
MOVIELENS_DIR = RAW_DIR / "ml-latest-small"

if not MOVIELENS_DIR.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    !wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip -O /content/ml.zip
    !unzip -q /content/ml.zip -d "{RAW_DIR}"

print(sorted(p.name for p in MOVIELENS_DIR.iterdir()))

['README.txt', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


In [5]:
import pandas as pd

ratings = pd.read_csv(MOVIELENS_DIR / "ratings.csv")
print(ratings.shape)

(100836, 4)


In [6]:
movies = pd.read_csv(MOVIELENS_DIR / "movies.csv")
links = pd.read_csv(MOVIELENS_DIR / "links.csv")
print(movies.shape, links.shape)

(9742, 3) (9742, 3)


In [7]:
from google.colab import userdata

TMDB_API_KEY = userdata.get("TMDB_API_KEY")
print(len(TMDB_API_KEY))

32


In [8]:
print(links["tmdbId"].isna().sum())

8


In [9]:
links = links.dropna(subset=["tmdbId"]).astype({"tmdbId": int})
print(links.shape)

(9734, 3)


In [10]:
rating_counts = ratings["movieId"].value_counts()
print((rating_counts < 5).sum())

6074


In [11]:
MIN_RATINGS_PER_MOVIE = 5

kept_movie_ids = rating_counts[rating_counts >= MIN_RATINGS_PER_MOVIE].index.intersection(links["movieId"])
ratings = ratings[ratings["movieId"].isin(kept_movie_ids)]
print(len(kept_movie_ids), ratings.shape)

3650 (90274, 4)


In [12]:
movies = movies[movies["movieId"].isin(kept_movie_ids)]
print(movies.shape)

(3650, 3)


In [13]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False)
print(movies["year"].isna().sum())

0


In [15]:
movies["year"] = movies["year"].astype(int)
print(movies.head(3))

   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  year  
0  Adventure|Animation|Children|Comedy|Fantasy  1995  
1                   Adventure|Children|Fantasy  1995  
2                               Comedy|Romance  1995  


In [16]:
import requests

TMDB_MOVIE_URL = "https://api.themoviedb.org/3/movie/{tmdb_id}"

response = requests.get(TMDB_MOVIE_URL.format(tmdb_id=862), params={"api_key": TMDB_API_KEY}, timeout=10)
print(response.status_code)
print(response.json().get("title"), response.json().get("poster_path"))

200
Toy Story /uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg


In [17]:
def fetch_tmdb_movie(tmdb_id: int) -> dict | None:
    response = requests.get(TMDB_MOVIE_URL.format(tmdb_id=tmdb_id), params={"api_key": TMDB_API_KEY}, timeout=10)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    movie = response.json()
    return {"poster_path": movie.get("poster_path"), "overview": movie.get("overview")}

print(fetch_tmdb_movie(862))
print(fetch_tmdb_movie(999999999))

{'poster_path': '/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg', 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."}
None


In [18]:
import json

TMDB_CACHE_PATH = RAW_DIR / "tmdb_cache.json"

tmdb_cache = json.loads(TMDB_CACHE_PATH.read_text()) if TMDB_CACHE_PATH.exists() else {}
tmdb_ids_to_fetch = [tmdb_id for tmdb_id in links.loc[links["movieId"].isin(kept_movie_ids), "tmdbId"] if str(tmdb_id) not in tmdb_cache]
print(len(tmdb_cache), len(tmdb_ids_to_fetch))

0 3650


In [19]:
import time

TMDB_SLEEP_SECONDS = 0.05
CACHE_SAVE_EVERY = 100

for i, tmdb_id in enumerate(tmdb_ids_to_fetch, start=1):
    tmdb_cache[str(tmdb_id)] = fetch_tmdb_movie(tmdb_id)
    time.sleep(TMDB_SLEEP_SECONDS)
    if i % CACHE_SAVE_EVERY == 0:
        TMDB_CACHE_PATH.write_text(json.dumps(tmdb_cache))
        print(i, "fetched")

TMDB_CACHE_PATH.write_text(json.dumps(tmdb_cache))
print("done:", len(tmdb_cache))

100 fetched
200 fetched
300 fetched
400 fetched
500 fetched
600 fetched
700 fetched
800 fetched
900 fetched
1000 fetched
1100 fetched
1200 fetched
1300 fetched
1400 fetched
1500 fetched
1600 fetched
1700 fetched
1800 fetched
1900 fetched
2000 fetched
2100 fetched
2200 fetched
2300 fetched
2400 fetched
2500 fetched
2600 fetched
2700 fetched
2800 fetched
2900 fetched
3000 fetched
3100 fetched
3200 fetched
3300 fetched
3400 fetched
3500 fetched
3600 fetched
done: 3650


In [20]:
tmdb_ids_without_poster = [tmdb_id for tmdb_id, meta in tmdb_cache.items() if meta is None or not meta["poster_path"]]
print(len(tmdb_ids_without_poster))

14


In [21]:
movie_ids_without_poster = links.loc[links["tmdbId"].astype(str).isin(tmdb_ids_without_poster), "movieId"]

movies = movies[~movies["movieId"].isin(movie_ids_without_poster)]
ratings = ratings[~ratings["movieId"].isin(movie_ids_without_poster)]
print(movies.shape, ratings.shape)

(3636, 4) (90111, 4)


In [22]:
user2idx = {int(user_id): idx for idx, user_id in enumerate(sorted(ratings["userId"].unique()))}
movie2idx = {int(movie_id): idx for idx, movie_id in enumerate(sorted(movies["movieId"]))}
print(len(user2idx), len(movie2idx))
print(list(movie2idx.items())[:3], list(movie2idx.items())[-1])

610 3636
[(1, 0), (2, 1), (3, 2)] (187595, 3635)


In [23]:
ratings["user_idx"] = ratings["userId"].map(user2idx)
ratings["movie_idx"] = ratings["movieId"].map(movie2idx)
print(ratings[["user_idx", "movie_idx"]].isna().sum().sum())
print(ratings.head(3))

0
   userId  movieId  rating  timestamp  user_idx  movie_idx
0       1        1     4.0  964982703         0          0
1       1        3     4.0  964981247         0          2
2       1        6     4.0  964982224         0          5


In [24]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

(PROCESSED_DIR / "user2idx.json").write_text(json.dumps(user2idx))
(PROCESSED_DIR / "movie2idx.json").write_text(json.dumps(movie2idx))
print(sorted(p.name for p in PROCESSED_DIR.iterdir()))

['movie2idx.json', 'user2idx.json']


In [25]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

ratings_train, ratings_rest = train_test_split(ratings, test_size=0.2, random_state=RANDOM_SEED)
ratings_val, ratings_test = train_test_split(ratings_rest, test_size=0.5, random_state=RANDOM_SEED)
print(len(ratings_train), len(ratings_val), len(ratings_test))

72088 9011 9012


In [26]:
for split_name, split in [("train", ratings_train), ("val", ratings_val), ("test", ratings_test)]:
    split.to_csv(PROCESSED_DIR / f"ratings_{split_name}.csv", index=False)

print(sorted(p.name for p in PROCESSED_DIR.iterdir()))

['movie2idx.json', 'ratings_test.csv', 'ratings_train.csv', 'ratings_val.csv', 'user2idx.json']


In [27]:
for split_name, split in [("train", ratings_train), ("val", ratings_val), ("test", ratings_test)]:
    loaded = pd.read_csv(PROCESSED_DIR / f"ratings_{split_name}.csv")
    assert len(loaded) == len(split), split_name
print("all splits load")

all splits load


In [28]:
movie_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean"])
print(movie_stats.sort_values("count", ascending=False).head(3))

         count      mean
movieId                 
356        329  4.164134
318        317  4.429022
296        307  4.197068


In [29]:
HOME_LIST_SIZE = 20
MIN_RATINGS_FOR_ALL_TIME = 50

popular_ids = movie_stats.sort_values("count", ascending=False).head(HOME_LIST_SIZE).index.tolist()
all_time_ids = (
    movie_stats[movie_stats["count"] >= MIN_RATINGS_FOR_ALL_TIME]
    .sort_values("mean", ascending=False)
    .head(HOME_LIST_SIZE)
    .index.tolist()
)
print(len(popular_ids), len(all_time_ids))
print(all_time_ids[:3])

20 20
[318, 858, 2959]


In [30]:
TMDB_TRENDING_URL = "https://api.themoviedb.org/3/trending/movie/week"
MIN_HOME_LIST_SIZE = 12

response = requests.get(TMDB_TRENDING_URL, params={"api_key": TMDB_API_KEY}, timeout=10)
response.raise_for_status()
trending_tmdb_ids = [movie["id"] for movie in response.json()["results"]]

tmdb_to_movie_id = dict(zip(links["tmdbId"], links["movieId"]))
trending_ids = [tmdb_to_movie_id[t] for t in trending_tmdb_ids if t in tmdb_to_movie_id and tmdb_to_movie_id[t] in movie2idx]
print("matched:", len(trending_ids))

if len(trending_ids) < MIN_HOME_LIST_SIZE:
    trending_ids += [movie_id for movie_id in popular_ids if movie_id not in trending_ids]
print(len(trending_ids))

matched: 0
20


In [31]:
import re

POSTER_BASE_URL = "https://image.tmdb.org/t/p/w342"
movie_id_to_tmdb = dict(zip(links["movieId"], links["tmdbId"]))

def build_movie_meta(movie) -> dict:
    tmdb_meta = tmdb_cache[str(movie_id_to_tmdb[movie.movieId])]
    return {
        "id": int(movie.movieId),
        "title": re.sub(r"\s*\(\d{4}\)\s*$", "", movie.title),
        "year": int(movie.year),
        "genres": movie.genres.split("|"),
        "poster": POSTER_BASE_URL + tmdb_meta["poster_path"],
        "overview": tmdb_meta["overview"],
    }

movies_meta = [build_movie_meta(movie) for movie in movies.itertuples()]
print(len(movies_meta))
print(movies_meta[0])

3636
{'id': 1, 'title': 'Toy Story', 'year': 1995, 'genres': ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'], 'poster': 'https://image.tmdb.org/t/p/w342/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg', 'overview': "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences."}


In [32]:
home_lists = {
    "all_time": [int(movie_id) for movie_id in all_time_ids],
    "popular": [int(movie_id) for movie_id in popular_ids],
    "trending": [int(movie_id) for movie_id in trending_ids],
}

assert all(movie["poster"] for movie in movies_meta), "a movie lacks a poster"
assert all(len(ids) >= MIN_HOME_LIST_SIZE for ids in home_lists.values()), "a home list is too short"

(PROCESSED_DIR / "movies_meta.json").write_text(json.dumps({"movies": movies_meta, "home_lists": home_lists}))
print("saved", len(movies_meta), {name: len(ids) for name, ids in home_lists.items()})

saved 3636 {'all_time': 20, 'popular': 20, 'trending': 20}
